In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    "/kaggle/input/siim-isic-melanoma-classification",
    "/kaggle/data/siim-isic-melanoma-classification",
    "/kaggle/input",
    "/kaggle/data",
]


def _find_file(fname: str) -> str:
    for d in DATA_DIR_CANDIDATES:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find {fname} in any of: {DATA_DIR_CANDIDATES}")


train_path = _find_file("train.csv")
test_path = _find_file("test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)


def _prep_meta(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["age_approx"] = pd.to_numeric(df["age_approx"], errors="coerce")
    for c in ["sex", "anatom_site_general_challenge"]:
        if c in df.columns:
            df[c] = (
                df[c].fillna("unknown").astype(str).str.strip().replace({"": "unknown"})
            )
        else:
            df[c] = "unknown"
    if "patient_id" not in df.columns:
        df["patient_id"] = "unknown"
    else:
        df["patient_id"] = df["patient_id"].fillna("unknown").astype(str)
    return df


train = _prep_meta(train)
test = _prep_meta(test)

y = train["target"].astype(float).values
global_mean = float(np.mean(y))


def _make_group_folds_balanced_stratified(
    df: pd.DataFrame, group_col: str, target_col: str, k: int = 5, seed: int = 42
):
    rng = np.random.RandomState(seed)

    g = df[group_col].astype(str)
    t = df[target_col].astype(float)

    g_stats = (
        pd.DataFrame({"g": g, "t": t})
        .groupby("g")["t"]
        .agg(["count", "sum"])
        .rename(columns={"count": "n", "sum": "pos"})
        .reset_index()
    )
    g_stats["rate"] = g_stats["pos"] / (g_stats["n"] + 1e-12)

    # Change (score-relevant): enforce deterministic ordering by descending positive-rate then descending size.
    # This preserves your stratified greedy objective but avoids subtle ordering mistakes that degrade OOF quality.
    jitter = rng.uniform(0, 1e-6, size=len(g_stats))
    g_stats = g_stats.sort_values(
        by=["rate", "n"], ascending=[False, False], kind="mergesort"
    ).reset_index(drop=True)
    if len(g_stats) > 1:
        tie_mask = g_stats[["rate", "n"]].duplicated(keep=False).to_numpy()
        if tie_mask.any():
            g_stats.loc[tie_mask, "_jit"] = jitter[tie_mask]
            g_stats = (
                g_stats.sort_values(
                    by=["rate", "n", "_jit"],
                    ascending=[False, False, True],
                    kind="mergesort",
                )
                .drop(columns=["_jit"])
                .reset_index(drop=True)
            )

    fold_groups = [[] for _ in range(k)]
    fold_n = np.zeros(k, dtype=np.float64)
    fold_pos = np.zeros(k, dtype=np.float64)

    total_n = float(g_stats["n"].sum())
    total_pos = float(g_stats["pos"].sum())
    target_n = total_n / k
    target_pos = total_pos / k

    for _, row in g_stats.iterrows():
        gn = float(row["n"])
        gp = float(row["pos"])

        best_j = None
        best_score = None
        for j in range(k):
            n_new = fold_n[j] + gn
            p_new = fold_pos[j] + gp
            dn = (n_new - target_n) / (target_n + 1e-12)
            dp = (p_new - target_pos) / (target_pos + 1e-12)
            score = dn * dn + dp * dp
            if (best_score is None) or (score < best_score):
                best_score = score
                best_j = j

        fold_groups[best_j].append(row["g"])
        fold_n[best_j] += gn
        fold_pos[best_j] += gp

    # Change (score-relevant bug-guard): build folds via a single group->fold mapping to guarantee disjoint indices.
    # Overlapping folds break OOF semantics and can collapse AUC toward random; this is a minimal correctness fix.
    group_to_fold = {}
    for j, fg in enumerate(fold_groups):
        for gg in fg:
            group_to_fold[str(gg)] = j

    groups = g.astype(str).to_numpy()
    fold_id = np.fromiter((group_to_fold.get(gg, -1) for gg in groups), dtype=np.int64)

    folds = []
    for j in range(k):
        folds.append(np.where(fold_id == j)[0])

    # Sanity: ensure full coverage and disjointness (deterministic repair if needed)
    used = np.concatenate(folds) if len(folds) else np.array([], dtype=np.int64)
    if used.size != len(np.unique(used)) or used.size != len(df):
        folds = [np.array([], dtype=np.int64) for _ in range(k)]
        for j in range(k):
            idx = np.where(fold_id == j)[0]
            folds[j] = idx
        # Any -1 groups (shouldn't happen) go to smallest fold
        miss = np.where(fold_id < 0)[0]
        if miss.size:
            lens = np.array([len(f) for f in folds])
            j0 = int(np.argmin(lens))
            folds[j0] = np.concatenate([folds[j0], miss])

    return folds


folds = _make_group_folds_balanced_stratified(
    train, "patient_id", "target", k=5, seed=42
)


def oof_target_encode_with_test_folds(
    train_df,
    test_df,
    col,
    folds,
    smoothing=10.0,
):
    """
    Returns:
      oof: (n_train,)
      te_avg: (n_test,) average of fold-specific encodings
      te_folds: list of (n_test,) fold-specific encodings (each built from that fold's train split)
    """
    oof = np.zeros(len(train_df), dtype=np.float64)
    all_idx = np.arange(len(train_df))

    te_folds = []
    te_accum = np.zeros(len(test_df), dtype=np.float64)

    for val_idx in folds:
        tr_idx = np.setdiff1d(all_idx, val_idx, assume_unique=False)
        tr = train_df.iloc[tr_idx]
        prior = float(tr["target"].mean())

        stats = tr.groupby(col)["target"].agg(["mean", "count"])
        smooth = (stats["count"] * stats["mean"] + smoothing * prior) / (
            stats["count"] + smoothing
        )

        oof[val_idx] = (
            train_df.iloc[val_idx][col]
            .map(smooth)
            .fillna(prior)
            .astype(np.float64)
            .values
        )

        te_fold = test_df[col].map(smooth).fillna(prior).astype(np.float64).values
        te_folds.append(te_fold)
        te_accum += te_fold

    te_avg = te_accum / float(len(folds))
    return oof, te_avg, te_folds


train_tmp = train[
    [
        "image_name",
        "patient_id",
        "target",
        "sex",
        "age_approx",
        "anatom_site_general_challenge",
    ]
].copy()
test_tmp = test[
    ["image_name", "patient_id", "sex", "age_approx", "anatom_site_general_challenge"]
].copy()

age_bins = [-np.inf, 20, 30, 40, 50, 60, 70, 80, np.inf]
train_tmp["age_bin"] = pd.cut(train_tmp["age_approx"], bins=age_bins).astype(str)
test_tmp["age_bin"] = pd.cut(test_tmp["age_approx"], bins=age_bins).astype(str)

train_tmp["sex_site"] = (
    train_tmp["sex"].astype(str)
    + "_"
    + train_tmp["anatom_site_general_challenge"].astype(str)
)
test_tmp["sex_site"] = (
    test_tmp["sex"].astype(str)
    + "_"
    + test_tmp["anatom_site_general_challenge"].astype(str)
)

train_tmp["site_age"] = (
    train_tmp["anatom_site_general_challenge"].astype(str)
    + "_"
    + train_tmp["age_bin"].astype(str)
)
test_tmp["site_age"] = (
    test_tmp["anatom_site_general_challenge"].astype(str)
    + "_"
    + test_tmp["age_bin"].astype(str)
)

train_tmp["sex_site_age"] = (
    train_tmp["sex_site"].astype(str) + "_" + train_tmp["age_bin"].astype(str)
)
test_tmp["sex_site_age"] = (
    test_tmp["sex_site"].astype(str) + "_" + test_tmp["age_bin"].astype(str)
)

train_tmp["sex_age"] = (
    train_tmp["sex"].astype(str) + "_" + train_tmp["age_bin"].astype(str)
)
test_tmp["sex_age"] = (
    test_tmp["sex"].astype(str) + "_" + test_tmp["age_bin"].astype(str)
)

train_tmp["patient_id_enc"] = train_tmp["patient_id"].astype(str)
test_tmp["patient_id_enc"] = test_tmp["patient_id"].astype(str)

train_tmp["site_sex_age"] = (
    train_tmp["anatom_site_general_challenge"].astype(str)
    + "_"
    + train_tmp["sex"].astype(str)
    + "_"
    + train_tmp["age_bin"].astype(str)
)
test_tmp["site_sex_age"] = (
    test_tmp["anatom_site_general_challenge"].astype(str)
    + "_"
    + test_tmp["sex"].astype(str)
    + "_"
    + test_tmp["age_bin"].astype(str)
)

TIE_NOISE = 0.0

oof1, te1, te1_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "anatom_site_general_challenge", folds=folds, smoothing=5.0
)
oof2, te2, te2_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "sex", folds=folds, smoothing=20.0
)
oof3, te3, te3_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "age_bin", folds=folds, smoothing=10.0
)
oof4, te4, te4_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "sex_site", folds=folds, smoothing=30.0
)
oof5, te5, te5_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "site_age", folds=folds, smoothing=30.0
)
oof6, te6, te6_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "sex_site_age", folds=folds, smoothing=40.0
)
oof7, te7, te7_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "sex_age", folds=folds, smoothing=25.0
)
oof8, te8, te8_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "patient_id_enc", folds=folds, smoothing=2000.0
)
oof9, te9, te9_f = oof_target_encode_with_test_folds(
    train_tmp, test_tmp, "site_sex_age", folds=folds, smoothing=50.0
)

if TIE_NOISE > 0:
    rng = np.random.RandomState(0)
    for arr in [oof1, oof2, oof3, oof4, oof5, oof6, oof7, oof8, oof9]:
        arr[:] = np.clip(arr + rng.normal(0, TIE_NOISE, size=arr.shape), 0, 1)
    for arr in [te1, te2, te3, te4, te5, te6, te7, te8, te9]:
        arr[:] = np.clip(arr + rng.normal(0, TIE_NOISE, size=arr.shape), 0, 1)

sub1 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te1})
sub2 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te2})
sub3 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te3})
sub4 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te4})
sub5 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te5})
sub6 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te6})
sub7 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te7})
sub8 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te8})
sub9 = pd.DataFrame({"image_name": test_tmp["image_name"].values, "target": te9})


def rank_data(sub: pd.DataFrame) -> pd.DataFrame:
    sub = sub.copy()
    sub["target"] = sub["target"].rank(method="first", pct=True).astype(np.float64)
    return sub


def rank_array(a: np.ndarray) -> np.ndarray:
    s = pd.Series(a)
    r = s.rank(method="first", pct=True).astype(np.float64).to_numpy()
    return r


oof_list = [oof1, oof2, oof3, oof4, oof5, oof6, oof7, oof8, oof9]

oof_ranked = [np.zeros_like(x, dtype=np.float64) for x in oof_list]
for val_idx in folds:
    for j, x in enumerate(oof_list):
        oof_ranked[j][val_idx] = rank_array(x[val_idx])

te_folds_list = [te1_f, te2_f, te3_f, te4_f, te5_f, te6_f, te7_f, te8_f, te9_f]
te_ranked = []
n_folds = len(folds)
for j in range(len(te_folds_list)):
    acc = np.zeros(len(test_tmp), dtype=np.float64)
    for f in range(n_folds):
        acc += rank_array(te_folds_list[j][f])
    te_ranked.append(acc / float(n_folds))

sub_list = []
for j, te_r in enumerate(te_ranked, start=1):
    sub_list.append(
        pd.DataFrame({"image_name": test_tmp["image_name"].values, f"target{j}": te_r})
    )

f_sub = sub_list[0]
for s in sub_list[1:]:
    f_sub = f_sub.merge(s, on="image_name")




In [2]:
def _sigmoid(z):
    z = np.clip(z, -35.0, 35.0)
    return 1.0 / (1.0 + np.exp(-z))


def _ridge_logistic_irls(X, y, lam=1e-2, iters=25):
    """
    Ridge-regularized logistic regression via IRLS.
    X: (n, p) includes intercept column if desired.
    y: (n,) in {0,1}
    Returns beta (p,)
    """
    n, p = X.shape
    beta = np.zeros(p, dtype=np.float64)

    reg = np.ones(p, dtype=np.float64)
    reg[0] = 0.0

    I = np.eye(p, dtype=np.float64)

    for _ in range(iters):
        eta = X @ beta
        p_hat = _sigmoid(eta)

        w = p_hat * (1.0 - p_hat)
        w = np.clip(w, 1e-6, None)
        z = eta + (y - p_hat) / w

        Xw = X * w[:, None]
        A = X.T @ Xw + lam * (I * reg)
        b = X.T @ (w * z)
        beta_new = np.linalg.solve(A, b)

        beta = 0.5 * beta + 0.5 * beta_new

    return beta


X = np.vstack(oof_ranked).T  # (n, p)
yy = y.astype(np.float64)

mu = X.mean(axis=0)
sd = X.std(axis=0)
sd = np.where(sd < 1e-12, 1.0, sd)
Xz = (X - mu) / sd

X1 = np.concatenate([np.ones((Xz.shape[0], 1), dtype=np.float64), Xz], axis=1)

lam = 5e-2
beta = _ridge_logistic_irls(X1, yy, lam=lam, iters=25)

Xt = f_sub[[c for c in f_sub.columns if c.startswith("target")]].to_numpy(
    dtype=np.float64
)
Xtz = (Xt - mu) / sd
Xt1 = np.concatenate([np.ones((Xtz.shape[0], 1), dtype=np.float64), Xtz], axis=1)

pred = _sigmoid(Xt1 @ beta)

stable_tie = (
    pd.Series(f_sub["image_name"])
    .astype(str)
    .apply(lambda s: int(s.split("_")[-1]))  # stable for ISIC_########
    .astype(np.float64)
    .to_numpy()
)
stable_tie = (stable_tie - stable_tie.min()) / (
    stable_tie.max() - stable_tie.min() + 1e-12
)
pred = pred + 1e-12 * stable_tie

pred = rank_array(pred)

f_sub_out = pd.DataFrame(
    {
        "image_name": f_sub["image_name"].values,
        "target": np.clip(pred, 0.0, 1.0).astype(np.float64),
    }
)

sample_path = _find_file("sample_submission.csv")
sample = pd.read_csv(sample_path)[["image_name"]]
f_sub_out = sample.merge(f_sub_out, on="image_name", how="left")
f_sub_out["target"] = f_sub_out["target"].fillna(global_mean).clip(0, 1)

f_sub_out.to_csv("blend_sub.csv", index=False)

w0 = float(beta[0])
w = beta[1:].astype(np.float64)

print(
    f"Saved submission: blend_sub.csv with shape {f_sub_out.shape} and columns {list(f_sub_out.columns)}"
)
print("Learned intercept and weights (target1..target9):")
print(w0, w.tolist())
print(f_sub_out.head())

Saved submission: blend_sub.csv with shape (4142, 2) and columns ['image_name', 'target']
Learned intercept and weights (target1..target9):
-4.218283699947296 [0.2807805632942471, 0.3052154135897641, 0.6243509857837455, 0.001254980468364972, 0.10259638737793861, -0.4009122423573013, -0.02067801829391301, -0.31399854624189444, 0.17328211451850434]
     image_name    target
0  ISIC_0052212  0.043699
1  ISIC_0076545  0.470304
2  ISIC_0085172  0.066876
3  ISIC_0086709  0.197006
4  ISIC_0109568  0.522936
